# Transfermarkt Citizenship Split Smoke Test

This notebook tests the new latest-transfers behavior where splitting is only by citizenship (no type split).

In [10]:
from pathlib import Path
import importlib.util
import sys

PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / 'src').exists():
    SRC_PATH = PROJECT_ROOT / 'src'
else:
    # If notebook is started from another working dir, try parent traversal
    SRC_PATH = (PROJECT_ROOT / '..' / '..' / 'src').resolve()

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

module_path = SRC_PATH / 'football_analytics' / 'scrapers' / 'transfermarkt.py'
module_name = 'tm_src_transfermarkt'
spec = importlib.util.spec_from_file_location(module_name, module_path)
if spec is None or spec.loader is None:
    raise RuntimeError(f'Could not load module from {module_path}')

tm_src = importlib.util.module_from_spec(spec)
sys.modules[module_name] = tm_src
spec.loader.exec_module(tm_src)

TransfermarktConfig = tm_src.TransfermarktConfig
TransfermarktScraper = tm_src.TransfermarktScraper

print('Using src path:', SRC_PATH)
print('Loaded transfermarkt module from:', module_path)

Using src path: C:\Users\david\Documents\Git\football-analytics\src
Loaded transfermarkt module from: C:\Users\david\Documents\Git\football-analytics\src\football_analytics\scrapers\transfermarkt.py


In [11]:
import inspect

tm_config_params = set(inspect.signature(TransfermarktConfig).parameters.keys())
print('TransfermarktConfig parameters include citizenship flag:', 'latest_transfers_split_by_citizenship' in tm_config_params)
print('TransfermarktConfig parameters include min_marktwert:', 'min_marktwert' in tm_config_params)

base_kwargs = {
    'latest_transfers_max_pages': '1',
    'dry_run': True,
    'max_retries': 3,
    'timeout_seconds': 30,
}
if 'min_marktwert' in tm_config_params:
    base_kwargs['min_marktwert'] = '0'

# Use keywords if supported by this runtime's dataclass signature.
if 'latest_transfers_split_by_citizenship' in tm_config_params:
    config = TransfermarktConfig(
        latest_transfers_split_by_citizenship=True,
        **base_kwargs,
    )
else:
    config = TransfermarktConfig(**base_kwargs)
    # Fallback for older versions where constructor does not expose this field.
    if hasattr(config, 'latest_transfers_split_by_citizenship'):
        setattr(config, 'latest_transfers_split_by_citizenship', True)
    else:
        raise RuntimeError(
            'This TransfermarktConfig version does not support latest_transfers_split_by_citizenship. '
            'Please restart kernel and re-run Cell 2 to import the updated src module.'
        )
    if hasattr(config, 'min_marktwert'):
        setattr(config, 'min_marktwert', '0')

scraper = TransfermarktScraper(config=config)
print('Scraper initialized with citizenship split:', scraper.config.latest_transfers_split_by_citizenship)
print('Configured min_marktwert:', getattr(scraper.config, 'min_marktwert', '<not supported>'))

TransfermarktConfig parameters include citizenship flag: True
TransfermarktConfig parameters include min_marktwert: True
Scraper initialized with citizenship split: True
Configured min_marktwert: 0


In [12]:
citizenships = scraper._fetch_citizenship_ids()
print('Total citizenship options:', len(citizenships))
print('Sample:', citizenships)

2026-06-04 11:05:43,726 | INFO | Found 254 citizenship options on latest transfers page.


Total citizenship options: 254
Sample: [(1, 'Afghanistan'), (3, 'Albania'), (4, 'Algeria'), (239, 'American Samoa'), (234, 'American Virgin Islands'), (5, 'Andorra'), (6, 'Angola'), (232, 'Anguilla'), (7, 'Antigua and Barbuda'), (9, 'Argentina'), (10, 'Armenia'), (233, 'Aruba'), (12, 'Australia'), (127, 'Austria'), (13, 'Azerbaijan'), (14, 'Bahamas'), (15, 'Bahrain'), (16, 'Bangladesh'), (17, 'Barbados'), (18, 'Belarus'), (19, 'Belgium'), (20, 'Belize'), (21, 'Benin'), (211, 'Bermuda'), (22, 'Bhutan'), (23, 'Bolivia'), (269, 'Bonaire'), (24, 'Bosnia-Herzegovina'), (25, 'Botswana'), (26, 'Brazil'), (276, 'British India'), (231, 'British Virgin Islands'), (27, 'Brunei Darussalam'), (28, 'Bulgaria'), (29, 'Burkina Faso'), (30, 'Burundi'), (79, 'Cambodia'), (31, 'Cameroon'), (80, 'Canada'), (32, 'Cape Verde'), (229, 'Cayman Islands'), (138, 'Central African Republic'), (171, 'Chad'), (33, 'Chile'), (34, 'China'), (164, 'Chinese Taipei'), (248, 'Christmas Island'), (83, 'Colombia'), (35, 'C

In [13]:
from urllib.parse import parse_qs, urlparse

base_url = scraper._with_min_marktwert(scraper.config.latest_transfers_url)
sample = citizenships[:5]
country_urls = [
    (scraper._with_query_params(base_url, {'land_id': str(country_id)}), country_name)
    for country_id, country_name in sample
]

# Core regression checks
assert all('leihe=' not in url for url, _ in country_urls), 'Found unexpected type-split parameter in URL'
assert all(parse_qs(urlparse(url).query).get('minMarktwert', [''])[0] == '0' for url, _ in country_urls), (
    'Expected minMarktwert=0 in all generated URLs'
    )

for url, country_name in country_urls:
    print(country_name, '->', url)

Afghanistan -> https://www.transfermarkt.com/statistik/neuestetransfers?minMarktwert=0&land_id=1
Albania -> https://www.transfermarkt.com/statistik/neuestetransfers?minMarktwert=0&land_id=3
Algeria -> https://www.transfermarkt.com/statistik/neuestetransfers?minMarktwert=0&land_id=4
American Samoa -> https://www.transfermarkt.com/statistik/neuestetransfers?minMarktwert=0&land_id=239
American Virgin Islands -> https://www.transfermarkt.com/statistik/neuestetransfers?minMarktwert=0&land_id=234


In [14]:
from urllib.parse import parse_qs, urlparse

if not citizenships:
    raise RuntimeError('No citizenship IDs found; cannot run smoke scrape.')

country_id, country_name = citizenships[1]
base_url = scraper._with_min_marktwert(scraper.config.latest_transfers_url)
test_url = scraper._with_query_params(base_url, {'land_id': str(country_id)})

query_params = parse_qs(urlparse(test_url).query)
assert query_params.get('minMarktwert', [''])[0] == '0', 'Expected minMarktwert=0 in smoke test URL'

pages, rows, upserted = scraper._scrape_latest_transfers_from_url(test_url)

print('Smoke scrape country:', country_name)
print('Smoke test URL:', test_url)
print({'pages': pages, 'rows': rows, 'upserted': upserted})
assert upserted == 0, 'Expected 0 upserts in dry-run mode'

2026-06-04 11:05:55,230 | INFO | Latest transfers [https://www.transfermarkt.com/statistik/neuestetransfers?minMarktwert=0&land_id=3] page 1 parsed: 25 rows
2026-06-04 11:05:55,230 | INFO | Dry run enabled; skipping latest transfers upserts.


Smoke scrape country: Albania
Smoke test URL: https://www.transfermarkt.com/statistik/neuestetransfers?minMarktwert=0&land_id=3
{'pages': 1, 'rows': 25, 'upserted': 0}


In [15]:
test_url

'https://www.transfermarkt.com/statistik/neuestetransfers?minMarktwert=0&land_id=3'

In [16]:
# Optional bounded end-to-end check of _scrape_latest_transfers without DB writes
subset = citizenships[:3]
scraper._fetch_citizenship_ids = lambda: subset
summary = scraper._scrape_latest_transfers()
print(summary)

2026-06-04 11:07:08,979 | INFO | Scraping latest transfers for citizenship: Afghanistan
2026-06-04 11:07:09,713 | INFO | Latest transfers [https://www.transfermarkt.com/statistik/neuestetransfers?minMarktwert=0&land_id=1] page 1 parsed: 5 rows
2026-06-04 11:07:09,714 | INFO | Dry run enabled; skipping latest transfers upserts.
2026-06-04 11:07:09,714 | INFO | Scraping latest transfers for citizenship: Albania
2026-06-04 11:07:16,278 | INFO | Latest transfers [https://www.transfermarkt.com/statistik/neuestetransfers?minMarktwert=0&land_id=3] page 1 parsed: 25 rows
2026-06-04 11:07:16,280 | INFO | Dry run enabled; skipping latest transfers upserts.
2026-06-04 11:07:16,280 | INFO | Scraping latest transfers for citizenship: Algeria
2026-06-04 11:07:23,752 | INFO | Latest transfers [https://www.transfermarkt.com/statistik/neuestetransfers?minMarktwert=0&land_id=4] page 1 parsed: 9 rows
2026-06-04 11:07:23,753 | INFO | Dry run enabled; skipping latest transfers upserts.


{'latest_transfers_pages_processed': 3, 'latest_transfers_rows_parsed': 39, 'latest_transfers_upserted': 0}
